In [ ]:
import pandas as pd
import numpy as np
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score , recall_score ,f1_score,precision_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from scipy.sparse import csr_matrix
import warnings
warnings.filterwarnings('ignore')



In [ ]:
df = pd.read_csv('..\\data\\processed\\ml-data.csv')


In [93]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 192807 entries, 0 to 192806
Data columns (total 6 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   id                  192807 non-null  int64 
 1   Chemical_Class      101473 non-null  object
 2   Habit_Forming       192807 non-null  object
 3   Therapeutic_Class   192807 non-null  object
 4   ingredients_mapped  192807 non-null  object
 5   side_mapped         189249 non-null  object
dtypes: int64(1), object(5)
memory usage: 8.8+ MB


In [94]:
df = df[["ingredients_mapped","side_mapped"]]

In [95]:
df = df.reset_index(drop=True)

In [96]:
df.head()

,ingredients_mapped,side_mapped
0,Haloperidol,agitation ## movement disorder ## sleep distur...
1,Bevacizumab,bleeding ## taste changes ## headache ## back ...
2,Darbepoetin alfa,abdominal pain ## high blood pressure ## skin ...
3,Darbepoetin alfa,abdominal pain ## high blood pressure ## skin ...
4,Darbepoetin alfa,abdominal pain ## high blood pressure ## skin ...


In [97]:
all_unique_values = set()

for cell in df['side_mapped'].dropna():
    if str(cell).strip() != "":
        terms = [t.strip() for t in str(cell).split('##') if t.strip()]
        all_unique_values.update(terms)

print(f"unique_values: {len(all_unique_values)}")

unique_values: 300


In [98]:
df.dropna(subset=["side_mapped"],inplace=True)

In [99]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 189249 entries, 0 to 192806
Data columns (total 2 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   ingredients_mapped  189249 non-null  object
 1   side_mapped         189249 non-null  object
dtypes: object(2)
memory usage: 4.3+ MB


In [100]:
df["side_mapped"] = (df["side_mapped"].astype(str).str.strip().str.replace(r'^##','',regex=True))

In [101]:
df["side_mapped"].str.strip().str.startswith('##').sum()

np.int64(0)

In [102]:
ingredients_series = df['ingredients_mapped'].apply(lambda x: [item.strip() for item in x.split('##')])


In [103]:
mlb = MultiLabelBinarizer()
encoded_array = mlb.fit_transform(ingredients_series)


In [104]:
ingredients_df = pd.DataFrame(
    encoded_array, 
    columns=mlb.classes_, 
    index=df.index
).astype("uint8")

In [105]:
ingredients_df.head()

,Abacavir,Abciximab,Abemaciclib,Abiraterone Acetate,Acalabrutinib,Acamprosate,Acarbose,Acebrophylline,Aceclofenac,Acenocoumarol,...,Zoledronic Acid,Zolmitriptan,Zolpidem,Zonisamide,Zopiclone,Zotepine,Zuclopenthixol,n-Butyl-2-cyanoacrylate,n-acetylcarnosine,sodium perborate monohydrate
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [106]:
df_ing = pd.concat([df, ingredients_df], axis=1)


In [107]:
mlb_side = MultiLabelBinarizer()

In [108]:
side_lists = df_ing['side_mapped'].apply(lambda x: [i.strip() for i in str(x).split('##')])
side_encoded_array = mlb_side.fit_transform(side_lists)

In [109]:
side_df = pd.DataFrame(side_encoded_array, columns=mlb_side.classes_, index=df_ing.index).astype("uint8")
side_df.head()


,abdominal distension,abdominal pain,abnormal blood clotting,abnormal blood count,abnormal blood test,abnormal calcium deposits,abnormal drug reaction,abnormal eye movements,abnormal lab test,abnormal lactation,...,vocal changes,vomiting,weight changes,weight gain,weight loss,white blood cells in urine,withdrawal syndrome,worsening mood,worsening respiratory status,yeast infection
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [110]:
df_full_features = pd.concat([df_ing, side_df], axis=1)

In [111]:
input_cols = list(mlb.classes_)
target_cols = list(mlb_side.classes_) 

In [112]:
target_cols = target_cols[1:] 

In [113]:
len(df_full_features)

189249

In [114]:
X = df_full_features[input_cols] 
y = df_full_features[target_cols] 

In [115]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=60)

In [116]:
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=60)

In [117]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(136731, 1656)
(24130, 1656)
(28388, 1656)


In [118]:
X_train = X_train.values if hasattr(X_train, 'values') else X_train
X_val   = X_val.values   if hasattr(X_val,   'values') else X_val
X_test  = X_test.values  if hasattr(X_test,  'values') else X_test
y_train = y_train.values if hasattr(y_train, 'values') else y_train
y_val   = y_val.values   if hasattr(y_val,   'values') else y_val
y_test  = y_test.values  if hasattr(y_test,  'values') else y_test

In [119]:
X_train = X_train.astype(np.float32)
X_val = X_val.astype(np.float32)
X_test = X_test.astype(np.float32)

In [120]:
valid_mask = (
    (y_train.sum(axis=0) > 0) & (y_train.sum(axis=0) < len(y_train)) &
    (y_val.sum(axis=0)   > 0) & (y_val.sum(axis=0)   < len(y_val))   &
    (y_test.sum(axis=0)  > 0) & (y_test.sum(axis=0)  < len(y_test)))
valid_target_indices = np.where(valid_mask)[0]
valid_target_names   = [target_cols[i] for i in valid_target_indices]

print(f"Targets before: {len(target_cols)}, after: {len(valid_target_names)}")

y_train = y_train[:,valid_target_indices]
y_val   = y_val[:,valid_target_indices]
y_test  = y_test[:,valid_target_indices]

Targets before: 299, after: 270


In [121]:
X_train_sparse = csr_matrix(X_train)
X_val_sparse   = csr_matrix(X_val)
X_test_sparse  = csr_matrix(X_test)

In [122]:
def evaluate_model(model, X, y_np, target_cols, dataset_name, sample_idx=0, top_n=15, threshold=0.10):

    probas = model.predict_proba(X)
    y_pred_proba = np.column_stack([probas[i][:, 1] for i in range(len(target_cols))])
    y_pred_binary = (y_pred_proba >= threshold).astype(int)
    auc_scores = roc_auc_score(y_np, y_pred_proba, average=None)
    recall_scores = recall_score(y_np, y_pred_binary, average=None, zero_division=0)
    ap_scores = average_precision_score(y_np, y_pred_proba, average=None)
    f1 = f1_score(y_np,  y_pred_binary, average=None, zero_division=0)

    auc_series    = pd.Series(auc_scores, index=target_cols).sort_values(ascending=False)
    recall_series = pd.Series(recall_scores, index=target_cols).sort_values(ascending=False)
    ap_series     = pd.Series(ap_scores,  index=target_cols).sort_values(ascending=False)
    f1_series     = pd.Series(f1, index=target_cols).sort_values(ascending=False)

    print(f"\n{'='*40}")
    print(f"Dataset: {dataset_name}")
    print(f"Mean AUC:       {auc_series.mean():.4f}")
    print(f"Mean Recall:    {recall_series.mean():.4f}")
    print(f"Mean Precision: {ap_series.mean():.4f}")
    print(f"Mean F1: {f1_series.mean():.4f}")
  
    predicted = pd.Series(y_pred_proba[sample_idx], index=target_cols).sort_values(ascending=False)
    top_effects = predicted[predicted >= threshold].head(top_n)
    actual_effects = pd.Series(y_np[sample_idx], index=target_cols)
    actual_effects = actual_effects[actual_effects == 1].index.tolist()
    hits = set(top_effects.index) & set(actual_effects)

    print(f"{'='*40}")
    print(f"Sample Index: {sample_idx}")
    print(f"\nPredicted Top Side Effects:")
    print(top_effects)
    print(f"Actual Side Effects: {actual_effects}")
    print(f"Correctly Predicted: {len(hits)} / {len(actual_effects)}")
    print(f"Hits: {list(hits)}")

    return auc_series, recall_series, ap_series, f1_series, top_effects, actual_effects


In [123]:
base_model = LGBMClassifier( 
    n_estimators=200,
    max_depth=3,
    colsample_bytree=0.2,
    min_child_samples=20,
    metric='auc',
    class_weight='balanced',  
    n_jobs=-1
)

In [124]:
lgb_model = MultiOutputClassifier(base_model, n_jobs=-1)
lgb_model.fit(X_train_sparse, y_train)


,estimator,"LGBMClassifie...00, n_jobs=-1)"
,n_jobs,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,3
,learning_rate,0.1
,n_estimators,200
,subsample_for_bin,200000
,objective,None
,class_weight,'balanced'
,min_split_gain,0.0


In [125]:
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.3,
    min_child_weight=10,
    reg_alpha=0.5,
    reg_lambda=0.5,
    use_label_encoder=False,
    eval_metric='auc',
    tree_method='approx', #approx ,hist
    base_score=0.5,    
    n_jobs=-1,
    random_state=42
)

In [126]:
xgb_model = MultiOutputClassifier(xgb_model, n_jobs=-1)
xgb_model.fit(X_train_sparse, y_train)

,estimator,"XGBClassifier...ree=None, ...)"
,n_jobs,-1
,objective,'binary:logistic'
,base_score,0.5
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.3
,device,None
,early_stopping_rounds,None


In [127]:
cat_base = CatBoostClassifier(
    iterations=400,        
    learning_rate=0.03,    
    depth=6,               
    subsample=0.8,
    reg_lambda=0.3,      
    eval_metric='AUC',
    verbose=0,
    random_state=42
)

In [129]:
cat_model = MultiOutputClassifier(cat_base, n_jobs=-1)
cat_model.fit(X_train_sparse, y_train)

,estimator,<catboost.cor...001680F0D0D70>
,n_jobs,-1


In [130]:
lr_base = LogisticRegression(
    C=1.0,                    
    solver='saga',           
    max_iter=1000,
    class_weight='balanced',  
    n_jobs=-1,
    random_state=42
)

In [131]:
lr_model = MultiOutputClassifier(lr_base, n_jobs=-1)
lr_model.fit(X_train_sparse, y_train)


,estimator,LogisticRegre...solver='saga')
,n_jobs,-1
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'saga'


In [132]:
results = {}

for name, mdl in [("LightGBM", lgb_model), 
                  ("XGBoost", xgb_model), 
                  ("CatBoost", cat_model),
                  ("LogReg", lr_model)]:
    
    probas       = mdl.predict_proba(X_val_sparse)
    y_pred_proba = np.column_stack([probas[i][:, 1] for i in range(len(valid_target_names))])
    y_pred_binary = (y_pred_proba >= 0.15).astype(int)
    
    auc    = roc_auc_score(y_val,    y_pred_proba,  average='macro')
    recall = recall_score(y_val,     y_pred_binary, average='macro', zero_division=0)
    ap = average_precision_score(y_val, y_pred_proba,  average='macro')
    f1 = f1_score(y_val,  y_pred_binary, average='macro', zero_division=0)
    
    results[name] = {'AUC': auc, 'Recall': recall, 'Precision': ap, 'F1': f1}

results_df = pd.DataFrame(results).T.sort_values('AUC', ascending=False)
print(results_df)

               AUC    Recall  Precision        F1
LogReg    0.995992  0.990438   0.724118  0.619163
CatBoost  0.994175  0.949770   0.935130  0.911245
LightGBM  0.974906  0.986483   0.749460  0.349116
XGBoost   0.876717  0.593255   0.615480  0.580931


In [133]:
for name, mdl in [("LogReg", lr_model), ("CatBoost", cat_model)]:
    probas       = mdl.predict_proba(X_train_sparse)
    y_pred_proba = np.column_stack([probas[i][:, 1] for i in range(len(valid_target_names))])
    auc = roc_auc_score(y_train, y_pred_proba, average='macro')
    print(f"{name} Train AUC: {auc:.4f}")

LogReg Train AUC: 0.9976
CatBoost Train AUC: 0.9973


In [137]:
probas = cat_model.predict_proba(X_val_sparse)
y_pred_proba = np.column_stack([probas[i][:, 1] for i in range(len(valid_target_names))])
y_pred_binary = (y_pred_proba >= 0.10).astype(int)

print(f"Average predicted positives per sample: {y_pred_binary.mean(axis=1).mean():.2f}")
print(f"Average actual positives per sample:    {y_val.mean(axis=1).mean():.2f}")

Average predicted positives per sample: 0.04
Average actual positives per sample:    0.03
